# IndiaStay Hotels — SQL Business Questions (Step 3)

Now we'll answer a similar (and bigger) set of questions using **SQL**
— the language almost every data/business analyst job expects you to
know, because most company data lives in databases, not CSV files.

### How this notebook works
We don't need a real database server for this. Python has a built-in
library called `sqlite3` that lets us create a lightweight database
file on our own computer. We'll:
1. Load our cleaned CSV into a SQL table
2. Write plain SQL queries against that table
3. Run each query and read the result

Each question below is written as: **the business question** →
**the SQL query** → **what the result tells us**.

## Step 0: Import tools

- `pandas` → still used to nicely display query results as a table
- `sqlite3` → Python's built-in mini SQL database engine

In [ ]:
import pandas as pd
import sqlite3

## Step 1: Load the cleaned data into a SQL table

We create a new SQLite database file called `indiastay.db`, then use
`df.to_sql(...)` to copy our cleaned CSV straight into it as a table
named `bookings`.

In [ ]:
df = pd.read_csv(
    "indiastay_hotel_bookings_clean.csv",
    parse_dates=["Booking_Date", "Check_In", "Check_Out"]
)

conn = sqlite3.connect("indiastay.db")
df.to_sql("bookings", conn, if_exists="replace", index=False)

print("Table 'bookings' created with", len(df), "rows")

## Step 2: A helper to run queries easily

Instead of writing `pd.read_sql_query(query, conn)` every time, we
make a tiny helper function `run()` that does it for us — it just
takes a SQL string and returns a clean table.

In [ ]:
def run(query):
    return pd.read_sql_query(query, conn)

# quick test
run("SELECT * FROM bookings LIMIT 5;")

## Section A: Cancellation Questions

### Q1. What is the overall cancellation rate?

`Is_Cancelled` is 1 for a cancelled booking and 0 otherwise, so
`AVG(Is_Cancelled)` directly gives us the cancellation rate.

In [ ]:
run('''
SELECT
    ROUND(AVG(Is_Cancelled) * 100, 1) AS cancellation_rate_percent
FROM bookings;
''')

### Q2. Which Room Type has the highest cancellation rate?

In [ ]:
run('''
SELECT
    Room_Type,
    COUNT(*) AS total_bookings,
    ROUND(AVG(Is_Cancelled) * 100, 1) AS cancellation_rate_percent
FROM bookings
GROUP BY Room_Type
ORDER BY cancellation_rate_percent DESC;
''')

### Q3. Which Booking Channel has the highest cancellation rate?

In [ ]:
run('''
SELECT
    Booking_Channel,
    COUNT(*) AS total_bookings,
    ROUND(AVG(Is_Cancelled) * 100, 1) AS cancellation_rate_percent
FROM bookings
GROUP BY Booking_Channel
ORDER BY cancellation_rate_percent DESC;
''')

### Q4. Which Customer Type cancels the most?

In [ ]:
run('''
SELECT
    Customer_Type,
    ROUND(AVG(Is_Cancelled) * 100, 1) AS cancellation_rate_percent
FROM bookings
GROUP BY Customer_Type
ORDER BY cancellation_rate_percent DESC;
''')

### Q5. Does lead time (days booked in advance) differ between
cancelled and non-cancelled bookings?

`CASE WHEN` in SQL works like an if/else — here it turns 0/1 into
readable labels for the group.

In [ ]:
run('''
SELECT
    CASE WHEN Is_Cancelled = 1 THEN 'Cancelled' ELSE 'Not Cancelled' END AS status,
    ROUND(AVG(Lead_Time_Days), 1) AS avg_lead_time_days
FROM bookings
GROUP BY Is_Cancelled;
''')

### Q6. Which city has the highest total number of cancellations?

In [ ]:
run('''
SELECT
    City,
    SUM(Is_Cancelled) AS total_cancellations
FROM bookings
GROUP BY City
ORDER BY total_cancellations DESC
LIMIT 10;
''')

### Q7. Which month has the highest cancellation rate?

In [ ]:
run('''
SELECT
    Booking_Month,
    ROUND(AVG(Is_Cancelled) * 100, 1) AS cancellation_rate_percent
FROM bookings
GROUP BY Booking_Month
ORDER BY cancellation_rate_percent DESC
LIMIT 5;
''')

**Section A takeaway:** cancellation rates stay close to ~18% no
matter how we slice it (room type, channel, customer type, lead
time) — there's no single "problem segment." The small differences by
channel (OTA/Walk-in a little higher) are the most actionable lead.

## Section B: Revenue Questions

### Q8. What is the total revenue, counting only non-cancelled bookings?

We filter with `WHERE Is_Cancelled = 0` because a cancelled booking
never actually earned any money.

In [ ]:
run('''
SELECT
    ROUND(SUM(Total_Amount), 0) AS total_revenue
FROM bookings
WHERE Is_Cancelled = 0;
''')

### Q9. What is the average booking value?

In [ ]:
run('''
SELECT
    ROUND(AVG(Total_Amount), 0) AS avg_booking_value
FROM bookings
WHERE Is_Cancelled = 0;
''')

### Q10. Which 10 cities generate the most revenue?

In [ ]:
run('''
SELECT
    City,
    ROUND(SUM(Total_Amount), 0) AS total_revenue
FROM bookings
WHERE Is_Cancelled = 0
GROUP BY City
ORDER BY total_revenue DESC
LIMIT 10;
''')

### Q11. Which Room Type generates the most total revenue?

In [ ]:
run('''
SELECT
    Room_Type,
    ROUND(SUM(Total_Amount), 0) AS total_revenue
FROM bookings
WHERE Is_Cancelled = 0
GROUP BY Room_Type
ORDER BY total_revenue DESC;
''')

### Q12. Which individual hotel (Hotel_ID) earns the most revenue?

In [ ]:
run('''
SELECT
    Hotel_ID,
    Hotel_Name,
    ROUND(SUM(Total_Amount), 0) AS total_revenue
FROM bookings
WHERE Is_Cancelled = 0
GROUP BY Hotel_ID
ORDER BY total_revenue DESC
LIMIT 10;
''')

### Q13. How does revenue trend across months?

In [ ]:
run('''
SELECT
    Booking_Month,
    ROUND(SUM(Total_Amount), 0) AS total_revenue
FROM bookings
WHERE Is_Cancelled = 0
GROUP BY Booking_Month
ORDER BY total_revenue DESC;
''')

### Q14. Does giving a bigger discount actually correlate with more bookings?

We check average discount alongside booking count, grouped into
discount "buckets."

In [ ]:
run('''
SELECT
    Discount_Percent,
    COUNT(*) AS num_bookings,
    ROUND(AVG(Total_Amount), 0) AS avg_revenue_per_booking
FROM bookings
WHERE Is_Cancelled = 0
GROUP BY Discount_Percent
ORDER BY Discount_Percent;
''')

**Section B takeaway:** revenue is spread fairly evenly across
cities (good — the business isn't overly dependent on one location),
but Suite and Premium rooms bring in disproportionately more revenue
relative to how many exist — a solid basis for an upsell
recommendation.

## Section C: Seasonality Questions

### Q15. Which month has the most bookings (peak season)?

In [ ]:
run('''
SELECT
    Booking_Month,
    COUNT(*) AS total_bookings
FROM bookings
GROUP BY Booking_Month
ORDER BY total_bookings DESC
LIMIT 3;
''')

### Q16. Which month has the fewest bookings (low season)?

In [ ]:
run('''
SELECT
    Booking_Month,
    COUNT(*) AS total_bookings
FROM bookings
GROUP BY Booking_Month
ORDER BY total_bookings ASC
LIMIT 3;
''')

### Q17. How do bookings differ by year (2024 vs 2025 vs 2026)?

In [ ]:
run('''
SELECT
    Booking_Year,
    COUNT(*) AS total_bookings
FROM bookings
GROUP BY Booking_Year
ORDER BY Booking_Year;
''')

**Section C takeaway:** bookings clearly drop from September through
November — that's the low season to target with promotions.

## Section D: Customer Behavior Questions

### Q18. Which booking channel is used the most?

In [ ]:
run('''
SELECT
    Booking_Channel,
    COUNT(*) AS total_bookings
FROM bookings
GROUP BY Booking_Channel
ORDER BY total_bookings DESC;
''')

### Q19. What is the most common payment method?

In [ ]:
run('''
SELECT
    Payment_Method,
    COUNT(*) AS total_bookings
FROM bookings
GROUP BY Payment_Method
ORDER BY total_bookings DESC;
''')

### Q20. What is the average number of guests per booking, by Customer Type?

In [ ]:
run('''
SELECT
    Customer_Type,
    ROUND(AVG(Guests), 2) AS avg_guests
FROM bookings
GROUP BY Customer_Type
ORDER BY avg_guests DESC;
''')

### Q21. Which Customer Type books the most nights on average (longest stays)?

In [ ]:
run('''
SELECT
    Customer_Type,
    ROUND(AVG(Nights), 2) AS avg_nights
FROM bookings
GROUP BY Customer_Type
ORDER BY avg_nights DESC;
''')

### Q22. What share of bookings actually completed a stay
(Checked-Out) vs. were Cancelled vs. No-Show?

In [ ]:
run('''
SELECT
    Booking_Status,
    COUNT(*) AS total_bookings,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM bookings), 1) AS percent_of_total
FROM bookings
GROUP BY Booking_Status
ORDER BY total_bookings DESC;
''')

**Section D takeaway:** no single channel or payment method dominates
— demand is spread across the board — and average group size/length
of stay stays fairly consistent across customer types.

## Step 3: Close the database connection

Good practice: always close the connection once you're done running
queries.

In [ ]:
conn.close()
print("Connection closed.")

## Summary

We answered 22 business questions purely in SQL, covering
cancellations, revenue, seasonality, and customer behavior — the
same four themes from our original business objective. The results
match what we found in the pandas EDA notebook, which is a good sign:
it means our cleaning and analysis are consistent no matter which
tool we use.

Next step: bring these same numbers into **Excel** to build a
KPI summary, before moving on to the Power BI dashboard.